<a href="https://colab.research.google.com/github/MthabisiPatrice/ML-Engineering/blob/main/Implementing_optimization_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step-by-step process to optimize ML models
This reading will guide you through the following steps:

Step 1: Set up the environment

Step 2: Apply model pruning

Step 3: Apply model quantization

Step 4: Perform hyperparameter tuning

Step 5: Evaluate the trade-offs

#Step 1: Set up the environment
Instructions
Create a new Jupyter notebook.  Start by setting up your Python 3.8 Azure ML kernel. Ensure you have the necessary libraries installed, including Scikit-Learn, TensorFlow, or PyTorch, depending on the model you're working with.

Setup commands

In [4]:
import subprocess
import sys

# Uninstall existing tensorflow to ensure a clean install of the desired version
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "tensorflow", "-y"])

# Install scikit-learn and a specific version of tensorflow compatible with tfmot
# tensorflow-model-optimization is compatible with Keras 2.x, which is bundled with TensorFlow <= 2.15.x
subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "tensorflow==2.15.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow-model-optimization"])

# The Keras version that comes with TensorFlow 2.15.0 is compatible.
# The `tf-keras` package is for Keras 3, which is not compatible with tensorflow-model-optimization,
# and the `conda install` command was failing anyway.


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'scikit-learn', 'tensorflow==2.15.0']' returned non-zero exit status 1.

Explanation
This setup ensures that you have the right tools to apply optimization techniques such as pruning and quantization.



#Step 2: Apply model pruning
Begin by applying model pruning to reduce the size of your model. In this step, you’ll remove unnecessary neurons or branches that have minimal impact on the overall accuracy.

Instructions
Train the model on a dataset.

Apply pruning by removing neurons or branches with the least contribution to model performance.

Example (Tensorflow)


In [5]:
import tensorflow_model_optimization as tfmot
import tensorflow as tf
from tensorflow.keras.datasets import mnist

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build a simple model
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Define pruning schedule
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.0, final_sparsity=0.5, begin_step=0, end_step=1000)
}

# Apply pruning to the model BEFORE compiling or training the base model.
# This wraps the layers of the base model with pruning functionality.
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

# Compile the pruned model
pruned_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the pruned model
# This training phase will apply pruning as per the schedule
callbacks = [tfmot.sparsity.keras.UpdatePruningStep(), tfmot.sparsity.keras.PruningSummaries(log_dir='/tmp/pruning_logs')]
pruned_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test), callbacks=callbacks)

# Strip pruning wrappers to remove pruning-specific layers and metadata
# This results in a smaller, dense model
final_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# The original 'model' compilation and training steps are removed as pruning is applied from the start.
# If the intention was to train a dense model first and then prune, a different approach would be needed,
# such as applying pruning to a new model instance that inherits weights or by using pruning-aware training from scratch.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: `prune_low_magnitude` can only prune an object of the following types: keras.models.Sequential, keras functional model, keras.layers.Layer, list of keras.layers.Layer. You passed an object of type: Sequential.

Explanation
Pruning reduces the size of the model and the computational resources required during inference without sacrificing much accuracy. This is especially useful for deploying models on resource-constrained devices.

#Step 3: Apply model quantization
Next, apply quantization to your model. This will reduce the precision of the model’s parameters, speeding up inference, especially on devices with limited computational resources.

Instructions
Quantize the model’s weights from 32-bit floating-point to 8-bit integers.

Evaluate the model’s performance after quantization to ensure that accuracy remains acceptable.

Example (Tensorflow lite quantization)

In [ ]:
# Convert the pruned model to a TensorFlow Lite quantized model
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()

Explanation
Quantization reduces the size of the model and speeds up inference by lowering the precision of its weights, which is particularly useful for deployment on mobile devices or edge devices.

#Step 4: Perform hyperparameter tuning
Perform hyperparameter tuning to find the optimal combination of parameters that maximize accuracy while maintaining reasonable response times.

Instructions
Use a grid search or a randomized search to explore different hyperparameter combinations.

Choose the parameters that offer the best balance between model performance and computational cost.

Example (grid search in Scikit-Learn)

In [ ]:
# Measure accuracy of the quantized model using the test set
interpreter = tf.lite.Interpreter(model_content=quantized_model)
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

# Evaluate accuracy
correct_predictions = 0
for i in range(len(x_test)):
    input_data = x_test[i:i+1].astype('float32')
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_index)
    predicted_label = output.argmax()
    if predicted_label == y_test[i]:
        correct_predictions += 1

accuracy = correct_predictions / len(x_test)
print(f'Quantized model accuracy: {accuracy * 100:.2f}%')

Explanation
Hyperparameter tuning helps you find the best settings for your model, improving accuracy while minimizing computational requirements.

#Step 5: Evaluate the trade-offs
Finally, evaluate the trade-offs between model complexity, accuracy, and response time. Document the changes in performance (e.g., accuracy and latency) before and after applying these optimization techniques.

Instructions
Compare the original model’s accuracy and response time with the optimized model.

Identify the key trade-offs. (For example, did pruning reduce accuracy but speed up inference significantly?)

Document your findings, and decide which optimizations are most suitable for your use case.

Explanation
Understanding the trade-offs between accuracy and speed helps you make informed decisions about which optimizations to apply in production environments.